### Turbine Health Metric and Lifetime Calculation

In [14]:
# -----------------------------
# Wind Turbine / Farm Health — relative (result / baseline)
# -----------------------------
import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Sequence, Dict, Optional, Tuple, Union
import ast

# -----------------------------
# Config (edit as needed)
# -----------------------------
baseline_folder = r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\baselines\TS_DA_HKN_full_filled_small_gaps_only"
result_folder   = [
    r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\results\HKN_TS_DA_HKN_Revenue_shutdown_1000",
    r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\results\HKN_TS_DA_HKN_Revenue_shutdown_1200",
    r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\results\HKN_TS_DA_HKN_Revenue_shutdown_1400",
    r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\results\HKN_TS_DA_HKN_Revenue_shutdown_1600"
]

for result_folder in result_folder:

    baseline_file = "turbine_metrics.parquet"
    result_file   = "turbine_metrics.parquet"
    # baseline_file = "turbine_timeseries_cum.parquet"
    # result_file   = "turbine_timeseries_cum.parquet"

    # combine path and file name
    baseline_path = Path(baseline_folder) / baseline_file
    result_path   = Path(result_folder)   / result_file

    # Health channels and weights (names MUST match level-0 'channel' in the input)
    health_channels: List[str] = [
        'RA_tbfa', 'RA_tbss', 'RA_TBM', 'RA_ttfa', 'RA_BRM', 'RA_ebrm',
        'RA_fbrm', 'RA_blade_torsion', 'RA_shaft_mx_mb_fixed',
        'RA_shaft_mz_mb_fixed', 'RA_ADC'
    ]
    health_weights: List[float] = [0.5, 0.5]  # used only when 'weighted' is requested

    # Turbine selection (None/[] = all eligible turbines)
    #include_turbines: Optional[Sequence[Union[str,int]]] = [1, 2, 3]
    include_turbines: Optional[Sequence[Union[str,int]]] = None
    # Turbine methods to compute (subset of {'weighted','mean','min','max'})
    turbine_channel_methods: Sequence[str] = ['max', 'min']

    # Farm aggregation:
    #   - Choose which turbine method feeds the farm aggregation (must be in turbine_channel_methods)
    farm_source_method: str = 'max'
    #   - Choose across-turbine aggregations (any of {'mean' | 'min' | 'max' | 'weighted_sum'})
    farm_agg_methods: Sequence[str] = ['mean', 'min', 'max']
    #   - Per-turbine weights (only used when 'weighted_sum' is in farm_agg_methods)
    turbine_weights: Optional[List[float]] = None

    # Output directory
    out_dir = Path(result_folder)

    # -----------------------------
    # Helpers: IO
    # -----------------------------
    _ID_CANDIDATES = ("id", "ID", "Id", "turbine_id", "TurbineId", "turbine", "Turbine")
    _TS_CANDIDATES = ("timestamp", "Timestamp", "time", "datetime", "DateTime")

    def _infer_col(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:
        for c in df.columns:
            for cand in candidates:
                if str(c).lower() == cand.lower():
                    return c
        return None

    def _nat_id_sort_key(val: str):
        """Sort turbine ids naturally: numeric ids (by value) first, then alphanumeric."""
        s = str(val)
        try:
            num = float(s)
            return (0, num)
        except (TypeError, ValueError):
            return (1, s.lower())

    def _ensure_dt(series: pd.Series) -> pd.Series:
        return pd.to_datetime(series, errors="coerce")

    def convert_id_column_to_multiindex_wide(
        df: pd.DataFrame,
        id_col: Optional[str] = None,
        timestamp_col: Optional[str] = None,
        agg: str = "first",
        keep_channels: Optional[List[str]] = None,  # if provided, also used as the desired channel order
    ) -> pd.DataFrame:
        """
        Convert a dataframe with an 'id' column + single-level channel columns into:
        - ('timestamp','')
        - (channel, turbine_id)
        Deterministic column order:
        - timestamp first
        - channels in keep_channels order (if provided) else alphabetical
        - turbine ids natural-sorted (2 before 10)
        """
        # --- detect id ---
        if id_col is None:
            id_col = _infer_col(df, _ID_CANDIDATES)
        if id_col is None:
            raise ValueError("Could not find a turbine id column among candidates: " + ", ".join(_ID_CANDIDATES))

        # --- detect or synthesize timestamp ---
        if timestamp_col is None or timestamp_col not in df.columns:
            timestamp_col = "timestamp"
            while timestamp_col in df.columns:
                timestamp_col += "_ts"
            df = df.copy()
            df[timestamp_col] = pd.NaT

        # --- determine channel columns to keep & order ---
        excluded = {timestamp_col, id_col}
        channel_cols_all = [c for c in df.columns if c not in excluded]
        if not channel_cols_all:
            raise ValueError("No channel columns found to convert.")

        if keep_channels is not None:
            keep_set = set(map(str, keep_channels))
            channel_cols = [c for c in channel_cols_all if str(c) in keep_set]
            if not channel_cols:
                raise ValueError(f"None of the requested channels are present: {sorted(keep_channels)}")
            channel_order = [str(c) for c in keep_channels if c in map(str, channel_cols)]
        else:
            channel_cols = channel_cols_all
            channel_order = sorted([str(c) for c in channel_cols])

        # --- tidy (melt → groupby → pivot) ---
        work = df[[timestamp_col, id_col] + channel_cols].copy()
        work[timestamp_col] = pd.to_datetime(work[timestamp_col], errors="coerce")
        for c in channel_cols:
            work[c] = pd.to_numeric(work[c], errors="coerce")

        long = work.melt(
            id_vars=[timestamp_col, id_col],
            value_vars=channel_cols,
            var_name="channel",
            value_name="value",
        )

        has_any_ts = long[timestamp_col].notna().any()

        if has_any_ts:
            # NORMAL MODE: one row per timestamp
            long_agg = (long
                        .groupby([timestamp_col, id_col, "channel"], dropna=False)["value"]
                        .agg(agg)
                        .reset_index())

            wide = long_agg.pivot_table(
                index=timestamp_col,
                columns=["channel", id_col],
                values="value",
                aggfunc=agg,
            )

            wide.columns = pd.MultiIndex.from_tuples(
                [(str(ch), str(tid)) for ch, tid in wide.columns.to_list()],
                names=["channel", "turbine_id"],
            )

            tids_present = sorted({tid for _, tid in wide.columns}, key=_nat_id_sort_key)
            ordered_cols = [(ch, tid) for ch in channel_order for tid in tids_present if (ch, tid) in wide.columns]

            wide = wide[ordered_cols].reset_index()
            wide.columns = pd.MultiIndex.from_tuples(
                [("timestamp", "")] + ordered_cols,
                names=["channel", "turbine_id"],
            )
            return wide

        # SNAPSHOT MODE: no valid timestamps → exactly one row
        snap = (long
                .groupby([id_col, "channel"], dropna=False)["value"]
                .agg(agg)
                .reset_index())

        s = snap.set_index(["channel", id_col])["value"]
        s.index = pd.MultiIndex.from_tuples(
            [(str(ch), str(tid)) for ch, tid in s.index.to_list()],
            names=["channel", "turbine_id"],
        )
        row = s.to_frame().T  # one row of data (no timestamp yet)

        tids_present = sorted({tid for _, tid in row.columns}, key=_nat_id_sort_key)
        ordered_cols = [(ch, tid) for ch in channel_order for tid in tids_present if (ch, tid) in row.columns]
        row = row.reindex(columns=ordered_cols)

        row.insert(0, ("timestamp", ""), pd.NaT)
        row.columns = pd.MultiIndex.from_tuples(row.columns, names=["channel", "turbine_id"])
        return row

    def load_parquet(path: str, health_channels: Optional[List[str]] = None) -> pd.DataFrame:
        """
        Returns a dataframe with 2-level MultiIndex columns:
        - ('timestamp','')
        - (channel, turbine_id)
        Handles:
        1) Already-correct MultiIndex
        2) Tuple-string columns like "('RA_tbfa','1')"
        3) Flat columns with 'id' + channel columns (timestamp optional; if missing -> NaT)
        """
        df = pd.read_parquet(path)

        # Case 1: already MultiIndex with 2 levels
        if isinstance(df.columns, pd.MultiIndex) and df.columns.nlevels == 2:
            return df

        # Case 2: columns serialized as strings of tuples -> try to parse
        try:
            cols = [ast.literal_eval(col) if isinstance(col, str) else col for col in df.columns]
            if all(isinstance(c, tuple) and len(c) == 2 for c in cols):
                df = df.copy()
                df.columns = pd.MultiIndex.from_tuples(cols, names=["channel", "turbine_id"])
                return df
        except Exception:
            pass  # fall-through to Case 3

        # Case 3: 'id' + channel columns -> convert (timestamp may be absent)
        converted = convert_id_column_to_multiindex_wide(
            df,
            id_col = _infer_col(df, _ID_CANDIDATES),
            timestamp_col = _infer_col(df, _TS_CANDIDATES),
            agg = "first",
            keep_channels = health_channels,  # None -> keep all
        )
        return converted

    def try_to_parquet(df: pd.DataFrame, path: Path) -> bool:
        try:
            df.to_parquet(path, index=False)
            return True
        except Exception:
            return False

    def _sanitize_label(s: str) -> str:
        """File-system friendly label."""
        return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in s)

    def save_outputs(
        dfs: Dict[str, pd.DataFrame],
        out_dir: Path,
        prefix: str = "",
    ) -> Dict[str, Path]:
        """
        Save multiple dataframes. Tries Parquet first, falls back to CSV.

        Parameters
        ----------
        dfs : mapping of short label -> DataFrame
            e.g. {"turbine_health": df1, "farm_health": df2, "relative": df3, ...}
        out_dir : Path
        prefix : str
            Optional filename prefix. If "", filenames are <label>.<ext>;
            otherwise <prefix>_<label>.<ext>.
        """
        out_dir.mkdir(parents=True, exist_ok=True)
        paths: Dict[str, Path] = {}

        def _make_name(label: str, ext: str) -> str:
            safe_label = _sanitize_label(label)
            if prefix:
                return f"{_sanitize_label(prefix)}_{safe_label}.{ext}"
            return f"{safe_label}.{ext}"

        for label, df in dfs.items():
            parq_path = out_dir / _make_name(label, "parquet")
            if try_to_parquet(df, parq_path):
                paths[f"{label}_parquet"] = parq_path
                continue
            csv_path = out_dir / _make_name(label, "csv")
            df.to_csv(csv_path, index=False)
            paths[f"{label}_csv"] = csv_path

        return paths

    # -----------------------------
    # Helpers: structure + validation
    # -----------------------------
    def _to_multiindex_cols(df: pd.DataFrame) -> pd.DataFrame:
        """Ensure columns are a pandas MultiIndex with 2 levels: (channel, turbine_id)."""
        if isinstance(df.columns, pd.MultiIndex):
            if df.columns.nlevels == 2:
                return df
            raise ValueError("Expected 2-level MultiIndex columns (channel, turbine_id).")
        if all(isinstance(c, tuple) and len(c) == 2 for c in df.columns):
            df = df.copy()
            df.columns = pd.MultiIndex.from_tuples(df.columns, names=["channel", "turbine_id"])
            return df
        raise ValueError("Columns must be a 2-level MultiIndex or tuples of (channel, turbine_id).")

    def _find_timestamp_column(df: pd.DataFrame) -> Tuple[str, str]:
        """Find timestamp column. Expected ('timestamp','') (case-insensitive for the first element)."""
        candidates = [('timestamp', ''), ('Timestamp', ''), ('time', ''), ('datetime', ''), ('DateTime', '')]
        for col in df.columns:
            if isinstance(col, tuple) and len(col) == 2:
                for cand, empty in candidates:
                    if str(col[0]).lower() == cand.lower() and str(col[1]) == empty:
                        return col
        raise ValueError("Could not locate timestamp column as ('timestamp','').")

    def _validate_channels_and_weights(
        df: pd.DataFrame,
        channels: List[str],
        weights: List[float],
        methods: Sequence[str],
    ) -> None:
        allowed = {'weighted','mean','min','max'}
        bad = [m for m in methods if m not in allowed]
        if bad:
            raise ValueError(f"Unsupported turbine methods: {bad}. Allowed: {sorted(allowed)}")

        df_channels = {str(c[0]) for c in df.columns if isinstance(c, tuple) and len(c) == 2}
        missing = [c for c in channels if c not in df_channels]
        if missing:
            raise ValueError(f"Missing health channels in relative frame: {missing}")

        if 'weighted' in methods:
            if len(channels) != len(weights):
                raise ValueError("health_channels and health_weights must have the same length for 'weighted'.")
            if not np.all(np.isfinite(weights)):
                raise ValueError("health_weights must be finite numbers.")
            if np.sum(weights) == 0:
                raise ValueError("Sum of health_weights is zero; cannot compute 'weighted' method.")

    def _list_turbines_with_all_channels(df: pd.DataFrame, channels: List[str]) -> List[str]:
        """Return turbine ids that include *all* requested channels (in this dataframe)."""
        want = set(channels)
        present: Dict[str, set] = {}
        for (ch, tid) in df.columns:
            if ch in want and tid != '':
                present.setdefault(str(tid), set()).add(ch)
        return sorted([tid for tid, chans in present.items() if want.issubset(chans)], key=lambda x: (len(x), x))

    def _choose_turbines(df: pd.DataFrame, channels: List[str], include: Optional[Sequence]) -> List[str]:
        complete = _list_turbines_with_all_channels(df, channels)
        if not include:
            if not complete:
                raise ValueError("No turbines found with all required channels in relative frame.")
            return complete
        inc = [str(t) for t in include]
        chosen = [t for t in inc if t in complete]
        if not chosen:
            raise ValueError("None of the requested turbines were found with all required channels in relative frame.")
        return chosen

    # -----------------------------
    # Build relative df, Logic: (result / baseline)
    # -----------------------------
    def build_relative_df_for_channels(
        df_baseline_raw: pd.DataFrame,
        df_result_raw: pd.DataFrame,
        health_channels: List[str],
    ) -> pd.DataFrame:
        """
        Build a wide relative dataframe with ('timestamp','') and ONLY the requested health_channels.
        Relative is computed as: q = result / baseline

        - If timestamps overlap -> align by common timestamps and compute relative.
        - If timestamps are NaN or there is no overlap -> fall back to line-by-line positional computation
        (use the first min(n_base, n_result) rows), keeping baseline's timestamp column (even if NaT).
        """
        dfb = _to_multiindex_cols(df_baseline_raw)
        dfr = _to_multiindex_cols(df_result_raw)

        tcol_b = _find_timestamp_column(dfb)
        tcol_r = _find_timestamp_column(dfr)

        # Determine turbines that have ALL requested channels in BOTH frames
        def turbines_with_all(df: pd.DataFrame, channels: List[str]) -> set:
            present: Dict[str, set] = {}
            for (ch, tid) in df.columns:
                if ch in channels and tid != '':
                    present.setdefault(str(tid), set()).add(ch)
            return {tid for tid, chans in present.items() if set(channels).issubset(chans)}

        tids_common = turbines_with_all(dfb, health_channels) & turbines_with_all(dfr, health_channels)
        if not tids_common:
            raise ValueError("No turbines have ALL requested channels in BOTH baseline and result.")

        data_cols = sorted([(ch, str(tid)) for ch in health_channels for tid in tids_common])

        # Ensure datetime dtype (tolerates NaT/NaN)
        ts_b = pd.to_datetime(dfb[tcol_b], errors='coerce')
        ts_r = pd.to_datetime(dfr[tcol_r], errors='coerce')

        # Try timestamp-based alignment if there is any overlap of valid timestamps
        valid_b = set(ts_b.dropna())
        valid_r = set(ts_r.dropna())
        overlap = sorted(valid_b.intersection(valid_r))

        if overlap:
            ts_idx = pd.Index(overlap)
            base_sel = (dfb.set_index(tcol_b)[data_cols].reindex(ts_idx).sort_index())
            res_sel  = (dfr.set_index(tcol_r)[data_cols].reindex(ts_idx).sort_index())

            rel = pd.DataFrame(index=ts_idx)
            for c in data_cols:
                b = pd.to_numeric(base_sel[c], errors='coerce')
                r = pd.to_numeric(res_sel[c],  errors='coerce')
                with np.errstate(divide='ignore', invalid='ignore'):
                    q = r / b
                rel[c] = q.replace([np.inf, -np.inf], np.nan)

            rel.reset_index(inplace=True)
            rel.rename(columns={'index': tcol_b}, inplace=True)
            rel.columns = pd.MultiIndex.from_tuples(rel.columns, names=["channel", "turbine_id"])
            return rel

        # Fallback: line-by-line positional computation (timestamp may be NaT/NaN)
        base_subset = dfb[[tcol_b] + data_cols].copy()
        res_subset  = dfr[[tcol_r] + data_cols].copy()

        n = min(len(base_subset), len(res_subset))
        if n == 0:
            raise ValueError("No rows available for positional relative computation.")

        base_subset = base_subset.iloc[:n].reset_index(drop=True)
        res_subset  = res_subset.iloc[:n].reset_index(drop=True)

        ts_out = pd.to_datetime(base_subset[tcol_b], errors='coerce')

        rel = pd.DataFrame()
        rel[('timestamp','')] = ts_out

        for c in data_cols:
            b = pd.to_numeric(base_subset[c], errors='coerce')
            r = pd.to_numeric(res_subset[c], errors='coerce')
            with np.errstate(divide='ignore', invalid='ignore'):
                q = r / b
            rel[c] = q.replace([np.inf, -np.inf], np.nan)

        rel = rel[[('timestamp','')] + data_cols]
        rel.columns = pd.MultiIndex.from_tuples(rel.columns, names=["channel", "turbine_id"])
        return rel

    # -----------------------------
    # Core: compute outputs on relative frame
    # -----------------------------
    def compute_turbine_and_farm_health_wide(
        df: pd.DataFrame,
        health_channels: List[str],
        health_weights: List[float],
        include_turbines: Optional[Sequence[Union[str,int]]],
        turbine_channel_methods: Sequence[str],
        farm_source_method: str,
        farm_agg_methods: Sequence[str],
        turbine_weights: Optional[Sequence[float]] = None,
    ) -> Dict[str, pd.DataFrame]:
        """
        Returns:
        - 'turbine_health_wide': columns MultiIndex:
                ('timestamp','') +
                ('health_<method>', <turbine_id>) for each method in turbine_channel_methods
        - 'farm_health_wide'   : ('timestamp','') + one column per requested farm aggregation:
                ('farm_<agg>', '')
            where each <agg> aggregates across turbines using per-turbine values from <farm_source_method>.
        """
        # Normalize structure and inputs
        df = _to_multiindex_cols(df)
        ts_col = _find_timestamp_column(df)

        methods = [m.lower() for m in turbine_channel_methods]
        _validate_channels_and_weights(df, health_channels, health_weights, methods)

        farm_method = farm_source_method.lower()
        if farm_method not in methods:
            raise ValueError(f"farm_source_method '{farm_source_method}' must be in turbine_channel_methods {methods}.")

        valid_farm_aggs = {'mean', 'min', 'max', 'weighted_sum'}
        farm_aggs = [a.lower() for a in farm_agg_methods]
        bad_aggs = [a for a in farm_aggs if a not in valid_farm_aggs]
        if bad_aggs:
            raise ValueError(f"Unsupported farm_agg_methods: {bad_aggs}. Allowed: {sorted(valid_farm_aggs)}")

        # Timestamp
        ts = df[ts_col]
        if not np.issubdtype(ts.dtype, np.datetime64):
            ts = pd.to_datetime(ts, errors='coerce')
        base = pd.DataFrame({('timestamp',''): ts}).dropna(subset=[('timestamp','')])

        # Turbine selection from RELATIVE frame (restricted to channels present)
        turbines = _choose_turbines(df, health_channels, include_turbines)

        # Helper to build channel matrix per turbine
        def _channel_matrix_for_tid(tid: str) -> np.ndarray:
            cols = [(ch, tid) for ch in health_channels]
            return df[cols].to_numpy(dtype=float)

        # Prepare per-method settings
        weights_arr = np.asarray(health_weights, dtype=float) if 'weighted' in methods else None

        # Build turbine health for all requested methods
        turbine_health_wide = base.copy()
        for tid in turbines:
            mat = _channel_matrix_for_tid(tid)
            for m in methods:
                if m == 'weighted':
                    vec = (mat * weights_arr).sum(axis=1)
                elif m == 'mean':
                    vec = np.nanmean(mat, axis=1)
                elif m == 'min':
                    vec = np.nanmin(mat, axis=1)
                elif m == 'max':
                    vec = np.nanmax(mat, axis=1)
                turbine_health_wide[(f'health_{m}', str(tid))] = vec

        if not isinstance(turbine_health_wide.columns, pd.MultiIndex):
            turbine_health_wide.columns = pd.MultiIndex.from_tuples(turbine_health_wide.columns)
        turbine_health_wide.columns = pd.MultiIndex.from_tuples(
            turbine_health_wide.columns, names=["channel", "turbine_id"]
        )

        # -----------------------------
        # Farm aggregations from chosen turbine method — MULTIPLE
        # -----------------------------
        src_cols = [(f'health_{farm_method}', str(t)) for t in turbines]
        health_mat = np.column_stack([turbine_health_wide[c].to_numpy() for c in src_cols])

        farm_health_wide = base.copy()
        for agg in farm_aggs:
            if agg == 'weighted_sum':
                if turbine_weights is None:
                    tw = np.ones(len(turbines), dtype=float)
                else:
                    tw = np.asarray(turbine_weights, dtype=float)
                    if len(tw) != len(turbines):
                        raise ValueError("turbine_weights length must match the number of selected turbines.")
                farm_vec = (health_mat * tw.reshape(1, -1)).sum(axis=1)
            elif agg == 'mean':
                farm_vec = np.nanmean(health_mat, axis=1)
            elif agg == 'min':
                farm_vec = np.nanmin(health_mat, axis=1)
            elif agg == 'max':
                farm_vec = np.nanmax(health_mat, axis=1)
            farm_health_wide[(f'farm_{agg}', '')] = farm_vec

        if not isinstance(farm_health_wide.columns, pd.MultiIndex):
            farm_health_wide.columns = pd.MultiIndex.from_tuples(farm_health_wide.columns)
        farm_health_wide.columns = pd.MultiIndex.from_tuples(
            farm_health_wide.columns, names=["channel", "turbine_id"]
        )

        return {
            "turbine_health_wide": turbine_health_wide,
            "farm_health_wide": farm_health_wide,
            "selected_turbines": turbines,
            "turbine_methods": methods,
            "farm_source_method": farm_method,
            "farm_agg_methods": farm_aggs,
        }

    def convert_to_lifetime(df: pd.DataFrame) -> pd.DataFrame:
        """
        Convert a wide-format dataframe of health metrics into lifetime metrics.

        Formula (core math, can be swapped later):
            lifetime = (1 - d) + 1
        """
        if not isinstance(df.columns, pd.MultiIndex) or df.columns.nlevels != 2:
            raise ValueError("Input must have MultiIndex columns with (channel, turbine_id).")

        ts_col = ('timestamp', '')
        if ts_col not in df.columns:
            raise ValueError("Missing ('timestamp','') column in input dataframe.")

        out = df.copy()
        metric_cols = [c for c in out.columns if c != ts_col]
        for c in metric_cols:
            d = pd.to_numeric(out[c], errors='coerce')
            with np.errstate(invalid='ignore'):
                lifetime = (1 - d) + 1
            out[c] = lifetime

        return out

    # -----------------------------
    # Run: load both, build relative, compute metrics (single outputs)
    # -----------------------------
    if __name__ == "__main__":
        # Load
        df_baseline_raw = load_parquet(baseline_path, health_channels=health_channels)
        df_result_raw   = load_parquet(result_path,   health_channels=health_channels)

        # Build RELATIVE dataframe only for requested health_channels (+ timestamp)
        df_relative = build_relative_df_for_channels(
            df_baseline_raw, df_result_raw, health_channels=health_channels
        )

        # Compute turbine + farm health on RELATIVE data
        res_relative = compute_turbine_and_farm_health_wide(
            df_relative,
            health_channels=health_channels,
            health_weights=health_weights,
            include_turbines=include_turbines,          # optional subset; must exist in df_relative
            turbine_channel_methods=turbine_channel_methods,
            farm_source_method=farm_source_method,
            farm_agg_methods=farm_agg_methods,
            turbine_weights=turbine_weights,
        )

        # Final outputs
        turbine_health_df = res_relative["turbine_health_wide"]
        farm_health_df    = res_relative["farm_health_wide"]

        # Convert into Lifetime
        relative_lifetime_df = convert_to_lifetime(df_relative)
        turbine_lifetime_df  = convert_to_lifetime(turbine_health_df)
        farm_lifetime_df     = convert_to_lifetime(farm_health_df)

        # Save all outputs
        paths = save_outputs(
            {
                "relative": df_relative,
                "turbine_health": turbine_health_df,
                "farm_health": farm_health_df,
                "relative_lifetime": relative_lifetime_df,
                "turbine_lifetime": turbine_lifetime_df,
                "farm_lifetime": farm_lifetime_df,
            },
            out_dir=out_dir,
            prefix="relative",
        )

        # (optional) peek
        print("Saved files:", {k: str(v) for k, v in paths.items()})
        print(turbine_health_df.head())
        print(farm_health_df.head())





C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragment

Saved files: {'relative_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1000\\relative_relative.parquet', 'turbine_health_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1000\\relative_turbine_health.parquet', 'farm_health_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1000\\relative_farm_health.parquet', 'relative_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1000\\relative_relative_lifetime.parquet', 'turbine_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1000\\relative_turbine_lifetime.parquet', 'farm_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1000\\relative_farm_lifetime.parq

C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragment

Saved files: {'relative_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1200\\relative_relative.parquet', 'turbine_health_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1200\\relative_turbine_health.parquet', 'farm_health_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1200\\relative_farm_health.parquet', 'relative_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1200\\relative_relative_lifetime.parquet', 'turbine_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1200\\relative_turbine_lifetime.parquet', 'farm_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1200\\relative_farm_lifetime.parq

C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragment

Saved files: {'relative_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1400\\relative_relative.parquet', 'turbine_health_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1400\\relative_turbine_health.parquet', 'farm_health_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1400\\relative_farm_health.parquet', 'relative_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1400\\relative_relative_lifetime.parquet', 'turbine_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1400\\relative_turbine_lifetime.parquet', 'farm_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1400\\relative_farm_lifetime.parq

C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

C:\Users\mograf\AppData\Local\Temp\ipykernel_13212\87375960.py:431: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragment

Saved files: {'relative_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1600\\relative_relative.parquet', 'turbine_health_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1600\\relative_turbine_health.parquet', 'farm_health_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1600\\relative_farm_health.parquet', 'relative_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1600\\relative_relative_lifetime.parquet', 'turbine_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1600\\relative_turbine_lifetime.parquet', 'farm_lifetime_parquet': 'M:\\Projects\\Cost Model\\VP_loadframework\\sudoco_task2.3\\results\\HKN_TS_DA_HKN_Revenue_shutdown_1600\\relative_farm_lifetime.parq

### Plotting

In [16]:
# --- Plot turbine_lifetime & farm_lifetime from multiple result folders (last-row only) ---

import pandas as pd
import numpy as np
from pathlib import Path
import ast
import plotly.graph_objects as go

# ========= CONFIG =========
result_folders = [
    r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\results\HKN_TS_DA_HKN_Revenue_shutdown_1000",
    r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\results\HKN_TS_DA_HKN_Revenue_shutdown_1200",
    r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\results\HKN_TS_DA_HKN_Revenue_shutdown_1400",
    r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\results\HKN_TS_DA_HKN_Revenue_shutdown_1600"
]

turbine_file_basename = "relative_turbine_lifetime"  # saved by your pipeline
farm_file_basename    = "relative_farm_lifetime"     # saved by your pipeline

# Plot look & feel (match your example)
marker_size   = 10
marker_symbol = "triangle-up"
plot_width    = 1200
plot_height   = 600
template      = "plotly_white"
# ==========================


# ---------- Helpers ----------
def _load_any(folder: str, base: str) -> pd.DataFrame:
    """
    Load a DataFrame saved by your pipeline, trying Parquet first, then CSV.
    Accepts both proper MultiIndex columns and tuple-string encoded columns.
    """
    p_parquet = Path(folder) / f"{base}.parquet"
    p_csv     = Path(folder) / f"{base}.csv"

    if p_parquet.exists():
        df = pd.read_parquet(p_parquet)
    elif p_csv.exists():
        df = pd.read_csv(p_csv)
    else:
        raise FileNotFoundError(f"Neither {p_parquet} nor {p_csv} found.")

    # If columns are strings of tuples like "('health_max','3')", parse them
    if not isinstance(df.columns, pd.MultiIndex):
        try:
            cols = []
            ok = True
            for c in df.columns:
                if isinstance(c, str) and c.startswith("(") and c.endswith(")"):
                    cols.append(ast.literal_eval(c))
                else:
                    ok = False
                    break
            if ok and all(isinstance(t, tuple) and len(t) == 2 for t in cols):
                df = df.copy()
                df.columns = pd.MultiIndex.from_tuples(cols, names=["channel", "turbine_id"])
        except Exception:
            pass

    # Ensure MultiIndex with ("channel","turbine_id"); if still flat, coerce
    if isinstance(df.columns, pd.MultiIndex) and df.columns.nlevels == 2:
        return df

    cols = []
    for c in df.columns:
        if str(c).lower() == "timestamp":
            cols.append(("timestamp", ""))
        else:
            cols.append((str(c), ""))
    df = df.copy()
    df.columns = pd.MultiIndex.from_tuples(cols, names=["channel", "turbine_id"])
    return df


def _last_row_only(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep only the last row for plotting. Drop ('timestamp','') if present.
    Returns a single-row DataFrame with MultiIndex columns.
    """
    ts_col = ("timestamp", "")
    if ts_col in df.columns:
        df = df.drop(columns=[ts_col])
    if len(df) == 0:
        raise ValueError("DataFrame has no rows.")
    last = df.tail(1).copy()

    # Ensure numeric where possible
    for c in last.columns:
        last[c] = pd.to_numeric(last[c], errors="coerce")
    return last


def _label_from_folder(folder: str) -> str:
    return Path(folder).name


# ---------- Load (last row only) ----------
turbine_last_by_scenario = {}  # scenario -> single-row df (MultiIndex cols)
farm_last_by_scenario    = {}  # scenario -> single-row df (MultiIndex cols)

for d in result_folders:
    label = _label_from_folder(d)

    # turbine lifetime (last row)
    df_turb = _load_any(d, turbine_file_basename)
    last_turb = _last_row_only(df_turb)
    turbine_last_by_scenario[label] = last_turb

    # farm lifetime (last row)
    df_farm = _load_any(d, farm_file_basename)
    last_farm = _last_row_only(df_farm)
    farm_last_by_scenario[label] = last_farm


# ---------- Figure A: Turbine lifetime (last row) ----------
# Expect columns like ('health_max','<tid>'), ('health_min','<tid>'), etc.
# One figure per available health_* method; x-axis = turbine IDs; trace per scenario.

all_cols = set()
for df1 in turbine_last_by_scenario.values():
    all_cols |= set(df1.columns)

methods  = sorted({ch for (ch, tid) in all_cols if str(ch).startswith("health_")})
turbines = sorted({tid for (ch, tid) in all_cols if tid != ""})

for method in methods:
    fig_t = go.Figure()
    x = turbines

    for scen, df1 in turbine_last_by_scenario.items():
        y = []
        for tid in x:
            c = (method, tid)
            val = df1[c].iloc[0] if c in df1.columns else np.nan
            y.append(val)

        fig_t.add_trace(go.Scatter(
            x=x,
            y=y,
            mode="markers",
            marker=dict(size=marker_size, symbol=marker_symbol),
            name=scen,
            showlegend=True,
        ))

    fig_t.update_layout(
        title=f"Turbine Lifetime (last row) — {method}",
        xaxis_title="Turbine ID",
        yaxis_title="Lifetime",
        template=template,
        width=plot_width,
        height=plot_height,
        showlegend=True,
        legend_title_text="Scenario",
    )
    fig_t.show()


# ---------- Figure B: Farm lifetime (last row) ----------
# Expect columns like ('farm_mean',''), ('farm_min',''), ('farm_max',''), ('farm_weighted_sum','')
# X-axis = farm aggregations; trace per scenario.

farm_aggs = sorted({ch for df1 in farm_last_by_scenario.values() for (ch, tid) in df1.columns if tid == ""})
x_labels = farm_aggs

fig_f = go.Figure()
for scen, df1 in farm_last_by_scenario.items():
    y = []
    for ch in x_labels:
        c = (ch, "")
        val = df1[c].iloc[0] if c in df1.columns else np.nan
        y.append(val)

    fig_f.add_trace(go.Scatter(
        x=x_labels,
        y=y,
        mode="markers",
        marker=dict(size=marker_size, symbol=marker_symbol),
        name=scen,
        showlegend=True,
    ))

fig_f.update_layout(
    title="Farm Lifetime (last row)",
    xaxis_title="Farm Aggregation",
    yaxis_title="Lifetime",
    template=template,
    width=plot_width,
    height=plot_height,
    showlegend=True,
    legend_title_text="Scenario",
    xaxis=dict(tickangle=-20),
)
fig_f.show()


### Layout Plot

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import ast
import plotly.graph_objects as go

def plot_farm_lifetime_map(
    layout_csv=r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\data\HKN_layout_subset_with_scaled.csv",
    result_folder=r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\results\HKN_TS_DA_HKN_Revenue_shutdown_1200",
    method="max",  # e.g. "max", "min", "mean", "weighted"
    lifetime_file_basename="relative_turbine_lifetime",  # produced by your pipeline
    # column names in the layout file (defaults match your sample)
    id_col="ID",
    x_col="x_scaled",
    y_col="y_scaled",
    # ID alignment: layout_id = lifetime_id + id_offset
    # Your case: layout IDs start at 1, lifetime IDs at 0 -> id_offset = -1
    id_offset=-1,
    marker_size=12,
    colorscale="Viridis",
    template="plotly_white",
    width=1000,
    height=800
):
    """
    Plot turbines as colored dots on the farm layout, using LAST ROW of lifetime data.
    Colors represent lifetime for the chosen method (e.g., 'max' -> 'health_max').
    """

    # ---------- helpers ----------
    def _load_any(folder: str, base: str) -> pd.DataFrame:
        p_parquet = Path(folder) / f"{base}.parquet"
        p_csv     = Path(folder) / f"{base}.csv"

        if p_parquet.exists():
            df = pd.read_parquet(p_parquet)
        elif p_csv.exists():
            df = pd.read_csv(p_csv)
        else:
            raise FileNotFoundError(f"Could not find {p_parquet} or {p_csv}")

        # Parse tuple-string columns -> MultiIndex
        if not isinstance(df.columns, pd.MultiIndex):
            try:
                parsed = [ast.literal_eval(c) if isinstance(c, str) and c.startswith("(") and c.endswith(")") else c
                          for c in df.columns]
                if all(isinstance(t, tuple) and len(t) == 2 for t in parsed):
                    df = df.copy()
                    df.columns = pd.MultiIndex.from_tuples(parsed, names=["channel", "turbine_id"])
            except Exception:
                pass

        # Coerce to MultiIndex if still flat
        if not isinstance(df.columns, pd.MultiIndex):
            cols = []
            for c in df.columns:
                if str(c).lower() == "timestamp":
                    cols.append(("timestamp", ""))
                else:
                    cols.append((str(c), ""))
            df = df.copy()
            df.columns = pd.MultiIndex.from_tuples(cols, names=["channel", "turbine_id"])
        return df

    def _last_row_values_for_method(df: pd.DataFrame, method_name: str) -> pd.Series:
        ts_col = ("timestamp", "")
        if ts_col in df.columns:
            df = df.drop(columns=[ts_col])

        label = f"health_{method_name.lower()}"
        last = df.tail(1).copy()
        for c in last.columns:
            last[c] = pd.to_numeric(last[c], errors="coerce")

        cols = [c for c in last.columns if c[0] == label and str(c[1]) != ""]
        if not cols:
            raise ValueError(f"No columns found for method '{label}'.")
        s = last[cols].iloc[0]
        # index becomes turbine_id strings from the lifetime data
        s.index = [str(c[1]) for c in s.index]
        s.name = label
        return s

    # ---------- load layout ----------
    layout_df = pd.read_csv(layout_csv)
    if id_col not in layout_df.columns or x_col not in layout_df.columns or y_col not in layout_df.columns:
        raise ValueError(
            f"Layout file must contain columns '{id_col}', '{x_col}', '{y_col}'. "
            f"Found: {list(layout_df.columns)}"
        )

    # ---------- load lifetime + take last row ----------
    life_df = _load_any(result_folder, lifetime_file_basename)
    life_series = _last_row_values_for_method(life_df, method)

    # ---------- align IDs (layout_id = lifetime_id + id_offset) ----------
    # Build a lookup dict from lifetime ids (int) -> value
    life_lookup = {}
    for k, v in life_series.items():
        try:
            life_lookup[int(k)] = float(v)
        except Exception:
            # ignore non-integer turbine ids
            continue

    # Map each layout ID to the corresponding lifetime id
    layout_df = layout_df.copy()
    layout_df["_lifetime_id"] = layout_df[id_col].astype(int) + id_offset
    layout_df["lifetime_val"] = layout_df["_lifetime_id"].map(life_lookup).astype(float)

    # ---------- plot ----------
    scen_label = Path(result_folder).name
    mlabel = f"health_{method.lower()}"

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=layout_df[x_col],
        y=layout_df[y_col],
        mode="markers",
        marker=dict(
            size=marker_size,
            color=layout_df["lifetime_val"],
            colorscale=colorscale,
            showscale=True,
            colorbar=dict(title=mlabel),
            line=dict(width=0),
        ),
        text=[f"Turbine {tid}" for tid in layout_df[id_col]],
        hovertemplate=(
            "Scenario: %{customdata[0]}<br>"
            "Turbine: %{text}<br>"
            f"{mlabel}: %{{marker.color:.4f}}<extra></extra>"
        ),
        customdata=np.column_stack([np.full(len(layout_df), scen_label)]),
        name=scen_label,
    ))

    fig.update_layout(
        title=f"Farm Layout — {scen_label} — {mlabel}",
        xaxis_title=x_col,
        yaxis_title=y_col,
        template=template,
        width=width,
        height=height,
    )
    # Equal aspect ratio so geometry isn't skewed
    fig.update_yaxes(scaleanchor="x", scaleratio=1)

    fig.show()


# ---------- Example ----------
plot_farm_lifetime_map(
    result_folder=r"M:\Projects\Cost Model\VP_loadframework\sudoco_task2.3\results\HKN_TS_DA_HKN_Revenue_shutdown_1600",
    method="min",
    id_offset=-1  # layout IDs = lifetime IDs + 1
)
